# Show, Attend and Tell — Local Training (Flickr8k, 4 models)

**For local PCs with a CUDA GPU.** Tested target: RTX 5070 Ti (16GB).

### What this trains
1. **Baseline** (no attention, ResNet-50 backbone)
2. **SAT — ResNet-50 + Attention** (the paper's architecture)
3. **ViT-B/16 + Attention** (transformer encoder)
4. **CLIP ViT-B/32 + Attention** (multimodal pretrained)

### What's different from the Kaggle version
- Single dataset (Flickr8k only) — fits comfortably in time/VRAM
- All package installs and data downloads handled in cell 1 (run once, works offline after)
- Robust path handling — works on Windows, Linux, Mac
- Mixed precision + larger batch size for the 5070 Ti
- Resumable: if a model is already trained, the notebook skips it
- All 5 fixes from earlier (beam search, multi-reference eval, etc.) preserved

### Estimated training time on RTX 5070 Ti
- Baseline      : ~15 min
- ResNet-50+Att : ~25 min
- ViT-B/16+Att  : ~35 min
- CLIP+Att      : ~25 min
- **Total**     : ~1.5–2 hours

---
## Table of Contents
1. [Setup & Dependencies](#1)
2. [Download Flickr8k](#2)
3. [Load Captions](#3)
4. [Vocabulary](#4)
5. [Dataset & DataLoader](#5)
6. [Models](#6)
7. [Training](#7)
8. [Evaluation](#8)
9. [Results Table](#9)
10. [Export Best Model](#10)

---
## 1. Setup & Dependencies

In [ ]:
# ── Install dependencies (one-time, internet required for first run only) ────
import subprocess, sys, os

PACKAGES = [
    "torch",                  # already installed if you ran any PyTorch before
    "torchvision",
    "nltk",
    "tqdm",
    "pillow",
    "pandas",
    "matplotlib",
    "ftfy",                   # CLIP dependency
    "regex",                  # CLIP dependency
    "openai-clip",            # CLIP encoder
]

# Install only what's missing (faster on subsequent runs)
def pip_install(pkgs):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs),
                       check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("pip install warning:", e.stderr.decode()[-300:] if e.stderr else "")

pip_install(PACKAGES)
print("Dependencies installed.")

In [ ]:
# ── NLTK data (offline-friendly: only downloads if missing) ──────────────────
import nltk
NLTK_NEEDED = [("tokenizers/punkt",   "punkt"),
               ("tokenizers/punkt_tab","punkt_tab"),
               ("corpora/wordnet",    "wordnet"),
               ("corpora/omw-1.4",    "omw-1.4")]
for path, pkg in NLTK_NEEDED:
    try:
        nltk.data.find(path)
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception as e:
            print(f"  [warn] could not download {pkg}: {e}")
print("NLTK data ready.")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, re, json, random, time
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import GradScaler, autocast

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device detection ──────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch :", torch.__version__)
print("Device  :", device)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU     : {name} ({vram:.1f} GB)")
else:
    print("WARNING: No CUDA GPU — training will be very slow on CPU.")

USE_AMP = torch.cuda.is_available()

---
## 2. Local Flickr8k path

Edit `DATASET_DIR` in the cell below to point to your local Flickr8k folder.

The folder should contain:
- `Images/` — all the `.jpg` files
- `captions.txt` — the captions CSV

If your folder uses different names (`Flickr8k_Dataset/`, `Flickr8k.token.txt`, etc.),
the cell will try a few common alternatives automatically.

In [ ]:
# ── Point these to your local Flickr8k folder ────────────────────────────────
# Expected layout:
#   DATASET_DIR/
#     Images/   (all .jpg files)
#     captions.txt
#
# On Windows, use a raw string (r"...") so backslashes work, e.g.
#     DATASET_DIR = r"C:\Users\you\Datasets\flickr8k"
# On Linux/Mac:
#     DATASET_DIR = "/home/you/Datasets/flickr8k"

DATASET_DIR = r"C:\Users\elite\Desktop\SAT_local\SAT_local\Flickr8K"

IMG_DIR  = os.path.join(DATASET_DIR, "Images")
CAP_FILE = os.path.join(DATASET_DIR, "captions.txt")

# ── Sanity checks with helpful errors ────────────────────────────────────────
if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(
        f"DATASET_DIR does not exist: {DATASET_DIR}\n"
        "Update DATASET_DIR above to point to your Flickr8k folder."
    )
if not os.path.isdir(IMG_DIR):
    # Try alternate capitalization that some Flickr8k versions use
    for alt in ("Flicker8k_Images", "images", "Flickr8k_Dataset", "Flicker8k_Dataset"):
        cand = os.path.join(DATASET_DIR, alt)
        if os.path.isdir(cand):
            IMG_DIR = cand
            break
    else:
        raise FileNotFoundError(
            f"Images folder not found.\n"
            f"Looked at: {os.path.join(DATASET_DIR, 'Images')}\n"
            "Make sure DATASET_DIR contains an 'Images/' folder with all .jpg files."
        )
if not os.path.isfile(CAP_FILE):
    # Some versions name it differently
    for alt in ("captions.csv", "Flickr8k.token.txt"):
        cand = os.path.join(DATASET_DIR, alt)
        if os.path.isfile(cand):
            CAP_FILE = cand
            break
    else:
        raise FileNotFoundError(
            f"Captions file not found.\n"
            f"Looked at: {os.path.join(DATASET_DIR, 'captions.txt')}\n"
            "Make sure DATASET_DIR contains a 'captions.txt' file."
        )

# Output directory next to this notebook
OUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "outputs"))
os.makedirs(OUT_DIR, exist_ok=True)

print("Dataset dir :", DATASET_DIR)
print("Images dir  :", IMG_DIR)
print("Captions    :", CAP_FILE)
print("Output dir  :", OUT_DIR)

---
## 3. Load Captions

In [ ]:
df_captions = pd.read_csv(CAP_FILE)
print("Columns:", df_captions.columns.tolist())
print(df_captions.head(3))

image_to_captions = {}
for _, row in df_captions.iterrows():
    img_name = str(row["image"]).strip()
    caption  = str(row["caption"]).strip()
    image_to_captions.setdefault(img_name, []).append(caption)

# Keep only images that actually exist
all_images = [n for n in image_to_captions.keys()
              if os.path.isfile(os.path.join(IMG_DIR, n))]
random.shuffle(all_images)

n = len(all_images)
train_images = all_images[:int(0.8 * n)]
val_images   = all_images[int(0.8 * n):int(0.9 * n)]
test_images  = all_images[int(0.9 * n):]

print(f"\nTotal images: {n:,}")
print(f"  Train : {len(train_images):,}")
print(f"  Val   : {len(val_images):,}")
print(f"  Test  : {len(test_images):,}")

---
## 4. Vocabulary

In [ ]:
FREQ_THRESHOLD = 3   # Flickr8k is small — keep a wider vocab

word_counts = Counter()
for img_name in train_images:
    for caption in image_to_captions[img_name]:
        words = re.sub(r"[^a-z ]", "", caption.lower()).split()
        word_counts.update(words)

word_to_idx = {"<pad>":0, "<start>":1, "<end>":2, "<unk>":3}
idx_to_word = {0:"<pad>", 1:"<start>", 2:"<end>", 3:"<unk>"}

for word, count in word_counts.items():
    if count >= FREQ_THRESHOLD:
        idx = len(word_to_idx)
        word_to_idx[word] = idx
        idx_to_word[idx]  = word

VOCAB_SIZE = len(word_to_idx)
PAD_IDX    = word_to_idx["<pad>"]
START_IDX  = word_to_idx["<start>"]
END_IDX    = word_to_idx["<end>"]
UNK_IDX    = word_to_idx["<unk>"]

print(f"Vocabulary size: {VOCAB_SIZE:,}")

def caption_to_ids(caption):
    words = re.sub(r"[^a-z ]", "", caption.lower()).split()
    return [word_to_idx.get(w, UNK_IDX) for w in words]

# Save vocab for the inference notebook
vocab_path = os.path.join(OUT_DIR, "vocab.json")
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump({"word_to_idx": word_to_idx,
               "idx_to_word": {str(k):v for k,v in idx_to_word.items()}}, f)
print(f"Vocab saved -> {vocab_path}")

---
## 5. Dataset & DataLoader

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class Flickr8kDataset(Dataset):
    def __init__(self, image_names, transform):
        self.transform = transform
        self.pairs = [(name, cap)
                      for name in image_names
                      for cap in image_to_captions[name]]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        name, caption = self.pairs[idx]
        try:
            img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224))
        img  = self.transform(img)
        ids  = [START_IDX] + caption_to_ids(caption) + [END_IDX]
        return img, torch.tensor(ids, dtype=torch.long)


def collate_batch(batch):
    images, captions = zip(*batch)
    return torch.stack(images), pad_sequence(captions, batch_first=True, padding_value=PAD_IDX)


# ── Settings tuned for RTX 5070 Ti (16GB VRAM) ────────────────────────────────
import platform
BATCH_SIZE  = 64
NUM_WORKERS = 0 if platform.system() == "Windows" else 4

train_loader = DataLoader(Flickr8kDataset(train_images, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_batch,
                          pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0)
val_loader   = DataLoader(Flickr8kDataset(val_images, eval_transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_batch,
                          pin_memory=True)
test_loader  = DataLoader(Flickr8kDataset(test_images, eval_transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_batch,
                          pin_memory=True)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader):,}")
print(f"Val batches  : {len(val_loader):,}")
print(f"Test batches : {len(test_loader):,}")

In [ ]:
# Quick visual check that data is loading correctly
images, captions = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Caption batch shape:", captions.shape)

inv_normalize = transforms.Compose([
    transforms.Normalize([0,0,0], [1/s for s in STD]),
    transforms.Normalize([-m for m in MEAN], [1,1,1]),
])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    img  = inv_normalize(images[i]).permute(1,2,0).clamp(0,1).numpy()
    ids  = captions[i].tolist()
    words = [idx_to_word[x] for x in ids if x not in (PAD_IDX, START_IDX, END_IDX)]
    ax.imshow(img)
    ax.set_title(" ".join(words), fontsize=7, wrap=True)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 6. Models

In [ ]:
# ── Attention ─────────────────────────────────────────────────────────────────
class Attention(nn.Module):
    def __init__(self, encoder_dim=512, decoder_dim=512, attention_dim=512):
        super().__init__()
        self.encoder_linear = nn.Linear(encoder_dim, attention_dim)
        self.decoder_linear = nn.Linear(decoder_dim, attention_dim)
        self.score_linear   = nn.Linear(attention_dim, 1)

    def forward(self, encoder_features, decoder_hidden):
        enc_part = self.encoder_linear(encoder_features)
        dec_part = self.decoder_linear(decoder_hidden).unsqueeze(1)
        energy   = self.score_linear(torch.tanh(enc_part + dec_part)).squeeze(2)
        alpha    = torch.softmax(energy, dim=1)
        context  = (encoder_features * alpha.unsqueeze(2)).sum(dim=1)
        return context, alpha


# ── Helper for safe pretrained-weight loading ────────────────────────────────
def _try_pretrained(builder, weights_enum):
    """Load pretrained weights; if offline / unavailable, fall back to random init."""
    try:
        return builder(weights=weights_enum)
    except Exception as e:
        print(f"  [warn] pretrained weights unavailable, using random init: {type(e).__name__}")
        return builder(weights=None)


# ── Encoder: ResNet-50 (SAT) ──────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=True):
        super().__init__()
        resnet = _try_pretrained(models.resnet50, models.ResNet50_Weights.IMAGENET1K_V1)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])
        self.pool   = nn.AdaptiveAvgPool2d((14, 14))
        self.linear = nn.Linear(2048, encoded_dim)
        for p in self.resnet.parameters(): p.requires_grad = False
        if fine_tune:
            for p in list(self.resnet.children())[-1].parameters():
                p.requires_grad = True

    def forward(self, images):
        feat = self.pool(self.resnet(images))
        B    = feat.shape[0]
        return self.linear(feat.permute(0,2,3,1).reshape(B, 196, 2048))


# ── Encoder: Baseline (no attention) ──────────────────────────────────────────
class BaselineEncoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=True):
        super().__init__()
        resnet = _try_pretrained(models.resnet50, models.ResNet50_Weights.IMAGENET1K_V1)
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.linear = nn.Linear(2048, encoded_dim)
        for p in self.resnet.parameters(): p.requires_grad = False
        if fine_tune:
            for p in list(self.resnet.children())[-2].parameters():
                p.requires_grad = True

    def forward(self, images):
        return self.linear(self.resnet(images).view(images.size(0), -1))


# ── Encoder: ViT-B/16 ─────────────────────────────────────────────────────────
class ViTEncoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=True):
        super().__init__()
        from torchvision.models import vit_b_16, ViT_B_16_Weights
        vit = _try_pretrained(vit_b_16, ViT_B_16_Weights.IMAGENET1K_V1)
        self.conv_proj   = vit.conv_proj
        self.class_token = vit.class_token
        self.encoder     = vit.encoder
        self.project     = nn.Linear(768, encoded_dim)
        for p in vit.parameters(): p.requires_grad = False
        if fine_tune:
            for block in vit.encoder.layers[-2:]:
                for p in block.parameters(): p.requires_grad = True

    def forward(self, images):
        B   = images.shape[0]
        x   = self.conv_proj(images).reshape(B, 768, -1).permute(0, 2, 1)
        cls = self.class_token.expand(B, -1, -1)
        x   = self.encoder(torch.cat([cls, x], dim=1))
        return self.project(x[:, 1:, :])


# ── Encoder: CLIP ViT-B/32 ────────────────────────────────────────────────────
class CLIPEncoder(nn.Module):
    def __init__(self, encoded_dim=512, fine_tune=False):
        super().__init__()
        try:
            import clip as _clip_lib
        except ImportError:
            raise RuntimeError("CLIP not installed. Run cell 1 first.")
        clip_model, _ = _clip_lib.load("ViT-B/32", device="cpu")
        self.visual      = clip_model.visual.float()
        self.encoded_dim = encoded_dim
        self.project     = nn.Linear(512, encoded_dim)
        for p in self.visual.parameters(): p.requires_grad = False
        if fine_tune:
            for p in self.visual.transformer.resblocks[-1].parameters():
                p.requires_grad = True
        self.register_buffer("in_mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("in_std",  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
        self.register_buffer("cl_mean", torch.tensor([0.48145466,0.4578275,0.40821073]).view(1,3,1,1))
        self.register_buffer("cl_std",  torch.tensor([0.26862954,0.26130258,0.27577711]).view(1,3,1,1))

    def _renorm(self, x):
        return (x * self.in_std + self.in_mean - self.cl_mean) / self.cl_std

    def forward(self, images):
        images = self._renorm(images)
        v, B   = self.visual, images.shape[0]
        x      = v.conv1(images).reshape(B, v.conv1.out_channels, -1).permute(0,2,1)
        cls    = v.class_embedding.view(1,1,-1).expand(B,-1,-1)
        x      = v.ln_pre(torch.cat([cls, x], dim=1) + v.positional_embedding)
        x      = v.transformer(x.permute(1,0,2)).permute(1,0,2)
        patches = v.ln_post(x[:, 1:, :])
        if v.proj is not None: patches = patches @ v.proj
        patches = self.project(patches)
        feat    = patches.reshape(B, 7, 7, self.encoded_dim).permute(0,3,1,2)
        feat    = F.interpolate(feat, size=(14,14), mode="bilinear", align_corners=False)
        return feat.permute(0,2,3,1).reshape(B, 196, self.encoded_dim)


# ── Decoder with attention + beam search ──────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, embed_dim, decoder_dim, vocab_size,
                 encoder_dim=512, attention_dim=512, dropout=0.5):
        super().__init__()
        self.decoder_dim = decoder_dim
        self.vocab_size  = vocab_size
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention   = Attention(encoder_dim, decoder_dim, attention_dim)
        self.lstm_cell   = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h      = nn.Linear(encoder_dim, decoder_dim)
        self.init_c      = nn.Linear(encoder_dim, decoder_dim)
        self.beta_gate   = nn.Linear(decoder_dim, encoder_dim)
        self.dropout     = nn.Dropout(dropout)
        self.fc          = nn.Linear(decoder_dim, vocab_size)

    def init_hidden_state(self, feats):
        mean = feats.mean(dim=1)
        return torch.tanh(self.init_h(mean)), torch.tanh(self.init_c(mean))

    def forward(self, encoder_features, captions):
        B, _, _ = encoder_features.shape
        T       = captions.size(1)
        embeddings  = self.embedding(captions)
        h, c        = self.init_hidden_state(encoder_features)
        predictions = torch.zeros(B, T-1, self.vocab_size, device=encoder_features.device)
        alphas      = torch.zeros(B, T-1, 196,             device=encoder_features.device)
        for t in range(T - 1):
            ctx, alpha = self.attention(encoder_features, h)
            beta       = torch.sigmoid(self.beta_gate(h))
            ctx        = beta * ctx
            h, c       = self.lstm_cell(torch.cat([embeddings[:, t], ctx], dim=1), (h, c))
            predictions[:, t] = self.fc(self.dropout(h))
            alphas[:, t]      = alpha
        return predictions, alphas

    def generate_caption(self, encoder_features, max_len=50, beam_size=3):
        h, c = self.init_hidden_state(encoder_features)
        beams, completed = [(0.0, [START_IDX], h, c, [])], []
        for _ in range(max_len):
            new_beams = []
            for score, seq, hp, cp, alps in beams:
                last = torch.tensor([seq[-1]], device=encoder_features.device)
                emb  = self.embedding(last)
                ctx, alpha = self.attention(encoder_features, hp)
                beta = torch.sigmoid(self.beta_gate(hp))
                ctx  = beta * ctx
                hn, cn = self.lstm_cell(torch.cat([emb, ctx], dim=1), (hp, cp))
                lp     = torch.log_softmax(self.fc(hn), dim=1)
                topk_s, topk_w = lp[0].topk(beam_size)
                for s, w in zip(topk_s.tolist(), topk_w.tolist()):
                    entry = (score+s, seq+[w], hn, cn, alps+[alpha.squeeze(0).cpu()])
                    (completed if w==END_IDX else new_beams).append(entry)
            if not new_beams: break
            new_beams.sort(key=lambda x: x[0], reverse=True)
            beams = new_beams[:beam_size]
        if not completed:
            completed = [(b[0],b[1],b[2],b[3],b[4]) for b in beams]
        best = max(completed, key=lambda x: x[0])
        words = [idx_to_word[w] for w in best[1][1:]
                 if w not in (END_IDX, PAD_IDX, START_IDX, UNK_IDX)
                 and idx_to_word.get(w,"<unk>") not in ("<pad>","<start>","<end>","<unk>")]
        return words, best[4]


# ── Decoder for baseline (no attention) ──────────────────────────────────────
class BaselineDecoder(nn.Module):
    def __init__(self, embed_dim, decoder_dim, vocab_size, encoder_dim=512, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm_cell = nn.LSTMCell(embed_dim, decoder_dim)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)
        self.init_c    = nn.Linear(encoder_dim, decoder_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(decoder_dim, vocab_size)

    def forward(self, feat, captions):
        embeddings  = self.embedding(captions)
        h = torch.tanh(self.init_h(feat))
        c = torch.tanh(self.init_c(feat))
        preds = torch.zeros(captions.size(0), captions.size(1)-1,
                            self.fc.out_features, device=feat.device)
        for t in range(captions.size(1) - 1):
            h, c        = self.lstm_cell(self.dropout(embeddings[:, t]), (h, c))
            preds[:, t] = self.fc(self.dropout(h))
        return preds

    def generate_caption(self, feat, max_len=50, beam_size=3):
        h = torch.tanh(self.init_h(feat))
        c = torch.tanh(self.init_c(feat))
        beams, completed = [(0.0, [START_IDX], h, c)], []
        for _ in range(max_len):
            new_beams = []
            for score, seq, hp, cp in beams:
                last = torch.tensor([seq[-1]], device=feat.device)
                emb  = self.embedding(last)
                hn, cn = self.lstm_cell(emb, (hp, cp))
                lp     = torch.log_softmax(self.fc(hn), dim=1)
                topk_s, topk_w = lp[0].topk(beam_size)
                for s, w in zip(topk_s.tolist(), topk_w.tolist()):
                    entry = (score+s, seq+[w], hn, cn)
                    (completed if w==END_IDX else new_beams).append(entry)
            if not new_beams: break
            new_beams.sort(key=lambda x: x[0], reverse=True)
            beams = new_beams[:beam_size]
        if not completed:
            completed = [(b[0],b[1],b[2],b[3]) for b in beams]
        best = max(completed, key=lambda x: x[0])
        words = [idx_to_word[w] for w in best[1][1:]
                 if w not in (END_IDX, PAD_IDX, START_IDX, UNK_IDX)
                 and idx_to_word.get(w,"<unk>") not in ("<pad>","<start>","<end>","<unk>")]
        return words


print("All model classes defined.")

---
## 7. Training

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

LAMBDA_REG = 1.0
GRAD_CLIP  = 5.0
MAX_EPOCHS = 20
PATIENCE   = 5
BEAM_SIZE  = 3


def compute_bleu4(references, hypotheses):
    smooth = SmoothingFunction().method1
    return corpus_bleu(
        [[r.split() for r in refs] for refs in references],
        [h.split() for h in hypotheses],
        weights=(0.25,0.25,0.25,0.25),
        smoothing_function=smooth,
    )


def train_one_model(enc, dec, has_attention, model_name, ckpt_file):
    scaler    = GradScaler(enabled=USE_AMP)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    optimizer = torch.optim.Adam([
        {"params": dec.parameters(), "lr": 4e-4},
        {"params": filter(lambda p: p.requires_grad, enc.parameters()), "lr": 1e-4},
    ])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_bleu4, patience_count = 0.0, 0
    train_losses, val_losses, bleu4_scores = [], [], []
    print(f"\n{'='*60}\nTraining {model_name}\n{'='*60}")

    for epoch in range(1, MAX_EPOCHS + 1):
        enc.train(); dec.train()
        total_train = 0.0
        t0 = time.time()

        for images, captions in tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False):
            images   = images.to(device, non_blocking=True)
            captions = captions.to(device, non_blocking=True)

            with autocast(enabled=USE_AMP):
                features = enc(images)
                if has_attention:
                    preds, alphas = dec(features, captions)
                else:
                    preds  = dec(features, captions)
                    alphas = None

                targets = captions[:, 1:]
                B, T, V = preds.shape
                loss = criterion(preds.reshape(B*T, V), targets.reshape(B*T))
                if has_attention and alphas is not None:
                    loss = loss + LAMBDA_REG * ((1.0 - alphas.sum(dim=1))**2).mean()

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(list(enc.parameters()) + list(dec.parameters()), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            total_train += loss.item()

        avg_train = total_train / len(train_loader)

        # Validation loss (full val set, teacher forcing)
        enc.eval(); dec.eval()
        total_val = 0.0
        with torch.no_grad():
            for images, captions in val_loader:
                images   = images.to(device, non_blocking=True)
                captions = captions.to(device, non_blocking=True)
                with autocast(enabled=USE_AMP):
                    features = enc(images)
                    if has_attention:
                        preds, alphas = dec(features, captions)
                    else:
                        preds = dec(features, captions); alphas = None
                    targets = captions[:, 1:]
                    B, T, V = preds.shape
                    loss = criterion(preds.reshape(B*T,V), targets.reshape(B*T))
                    if has_attention and alphas is not None:
                        loss = loss + LAMBDA_REG * ((1.0 - alphas.sum(dim=1))**2).mean()
                total_val += loss.item()
        avg_val = total_val / len(val_loader)

        # Val BLEU-4 — full val set, all 5 references per image
        refs_list, hyps_list = [], []
        with torch.no_grad():
            for img_name in val_images:
                try:
                    raw = Image.open(os.path.join(IMG_DIR, img_name)).convert("RGB")
                except Exception:
                    continue
                tensor = eval_transform(raw).unsqueeze(0).to(device)
                feat   = enc(tensor)
                if has_attention:
                    gen, _ = dec.generate_caption(feat, beam_size=BEAM_SIZE)
                else:
                    gen    = dec.generate_caption(feat, beam_size=BEAM_SIZE)
                hyps_list.append(" ".join(gen))
                refs_list.append([re.sub(r"[^a-z ]", "", c.lower()).strip()
                                  for c in image_to_captions[img_name]])

        bleu4   = compute_bleu4(refs_list, hyps_list) if hyps_list else 0.0
        elapsed = time.time() - t0

        train_losses.append(avg_train)
        val_losses.append(avg_val)
        bleu4_scores.append(bleu4)
        scheduler.step(bleu4)

        print(f"Epoch {epoch:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
              f"BLEU-4: {bleu4:.4f} | {elapsed:.0f}s")

        if epoch % 2 == 0:
            sample_name = random.choice(val_images)
            try:
                raw  = Image.open(os.path.join(IMG_DIR, sample_name)).convert("RGB")
                feat = enc(eval_transform(raw).unsqueeze(0).to(device))
                w = (dec.generate_caption(feat, beam_size=BEAM_SIZE)[0] if has_attention
                     else dec.generate_caption(feat, beam_size=BEAM_SIZE))
                print("  Sample:", " ".join(w))
            except Exception:
                pass

        if bleu4 > best_bleu4:
            best_bleu4, patience_count = bleu4, 0
            torch.save({
                "encoder_state": enc.state_dict(),
                "decoder_state": dec.state_dict(),
                "has_attention": has_attention,
                "bleu4":         bleu4,
                "model_name":    model_name,
                "vocab_size":    VOCAB_SIZE,
            }, ckpt_file)
            print(f"  -> Best saved (BLEU-4 = {bleu4:.4f})")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    return train_losses, val_losses, bleu4_scores

In [ ]:
# ── Baseline (no att) ──────────────────────────────────────────────────────────────
base_enc = BaselineEncoder(512,True).to(device)
base_dec = BaselineDecoder(256,512,VOCAB_SIZE).to(device)
CKPT_BASE  = os.path.join(OUT_DIR, "best_base.pth")
_curve_path = os.path.join(OUT_DIR, "base_curves.json")

if os.path.isfile(CKPT_BASE) and os.path.isfile(_curve_path):
    try:
        print("Loading saved Baseline (no att)...")
        ckpt = torch.load(CKPT_BASE, map_location=device, weights_only=False)
        base_enc.load_state_dict(ckpt["encoder_state"])
        base_dec.load_state_dict(ckpt["decoder_state"])
        with open(_curve_path) as f:
            _c = json.load(f)
        base_train_losses = _c["train"]; base_val_losses = _c["val"]; base_bleu = _c["bleu4"]
    except Exception as _ckpt_err:
        print(f"Checkpoint incompatible ({_ckpt_err}), retraining Baseline (no att)...")
        base_train_losses, base_val_losses, base_bleu = train_one_model(
            base_enc, base_dec, False, "Baseline (no att)", CKPT_BASE
        )
        with open(_curve_path, "w") as f:
            json.dump({"train": base_train_losses, "val": base_val_losses, "bleu4": base_bleu}, f)
else:
    base_train_losses, base_val_losses, base_bleu = train_one_model(
        base_enc, base_dec, False, "Baseline (no att)", CKPT_BASE
    )
    with open(_curve_path, "w") as f:
        json.dump({"train": base_train_losses, "val": base_val_losses, "bleu4": base_bleu}, f)


In [ ]:
# ── ResNet-50 + Att ──────────────────────────────────────────────────────────────
att_enc = Encoder(512,True).to(device)
att_dec = Decoder(256,512,VOCAB_SIZE).to(device)
CKPT_ATT  = os.path.join(OUT_DIR, "best_att.pth")
_curve_path = os.path.join(OUT_DIR, "att_curves.json")

if os.path.isfile(CKPT_ATT) and os.path.isfile(_curve_path):
    try:
        print("Loading saved ResNet-50 + Att...")
        ckpt = torch.load(CKPT_ATT, map_location=device, weights_only=False)
        att_enc.load_state_dict(ckpt["encoder_state"])
        att_dec.load_state_dict(ckpt["decoder_state"])
        with open(_curve_path) as f:
            _c = json.load(f)
        att_train_losses = _c["train"]; att_val_losses = _c["val"]; att_bleu = _c["bleu4"]
    except Exception as _ckpt_err:
        print(f"Checkpoint incompatible ({_ckpt_err}), retraining ResNet-50 + Att...")
        att_train_losses, att_val_losses, att_bleu = train_one_model(
            att_enc, att_dec, True, "ResNet-50 + Att", CKPT_ATT
        )
        with open(_curve_path, "w") as f:
            json.dump({"train": att_train_losses, "val": att_val_losses, "bleu4": att_bleu}, f)
else:
    att_train_losses, att_val_losses, att_bleu = train_one_model(
        att_enc, att_dec, True, "ResNet-50 + Att", CKPT_ATT
    )
    with open(_curve_path, "w") as f:
        json.dump({"train": att_train_losses, "val": att_val_losses, "bleu4": att_bleu}, f)


In [ ]:
# ── ViT-B/16 + Att ──────────────────────────────────────────────────────────────
vit_enc = ViTEncoder(512,True).to(device)
vit_dec = Decoder(256,512,VOCAB_SIZE).to(device)
CKPT_VIT  = os.path.join(OUT_DIR, "best_vit.pth")
_curve_path = os.path.join(OUT_DIR, "vit_curves.json")

if os.path.isfile(CKPT_VIT) and os.path.isfile(_curve_path):
    try:
        print("Loading saved ViT-B/16 + Att...")
        ckpt = torch.load(CKPT_VIT, map_location=device, weights_only=False)
        vit_enc.load_state_dict(ckpt["encoder_state"])
        vit_dec.load_state_dict(ckpt["decoder_state"])
        with open(_curve_path) as f:
            _c = json.load(f)
        vit_train_losses = _c["train"]; vit_val_losses = _c["val"]; vit_bleu = _c["bleu4"]
    except Exception as _ckpt_err:
        print(f"Checkpoint incompatible ({_ckpt_err}), retraining ViT-B/16 + Att...")
        vit_train_losses, vit_val_losses, vit_bleu = train_one_model(
            vit_enc, vit_dec, True, "ViT-B/16 + Att", CKPT_VIT
        )
        with open(_curve_path, "w") as f:
            json.dump({"train": vit_train_losses, "val": vit_val_losses, "bleu4": vit_bleu}, f)
else:
    vit_train_losses, vit_val_losses, vit_bleu = train_one_model(
        vit_enc, vit_dec, True, "ViT-B/16 + Att", CKPT_VIT
    )
    with open(_curve_path, "w") as f:
        json.dump({"train": vit_train_losses, "val": vit_val_losses, "bleu4": vit_bleu}, f)


In [ ]:
# ── CLIP + Att ──────────────────────────────────────────────────────────────
clip_enc = CLIPEncoder(512,False).to(device)
clip_dec = Decoder(256,512,VOCAB_SIZE).to(device)
CKPT_CLIP  = os.path.join(OUT_DIR, "best_clip.pth")
_curve_path = os.path.join(OUT_DIR, "clip_curves.json")

if os.path.isfile(CKPT_CLIP) and os.path.isfile(_curve_path):
    try:
        print("Loading saved CLIP + Att...")
        ckpt = torch.load(CKPT_CLIP, map_location=device, weights_only=False)
        clip_enc.load_state_dict(ckpt["encoder_state"])
        clip_dec.load_state_dict(ckpt["decoder_state"])
        with open(_curve_path) as f:
            _c = json.load(f)
        clip_train_losses = _c["train"]; clip_val_losses = _c["val"]; clip_bleu = _c["bleu4"]
    except Exception as _ckpt_err:
        print(f"Checkpoint incompatible ({_ckpt_err}), retraining CLIP + Att...")
        clip_train_losses, clip_val_losses, clip_bleu = train_one_model(
            clip_enc, clip_dec, True, "CLIP + Att", CKPT_CLIP
        )
        with open(_curve_path, "w") as f:
            json.dump({"train": clip_train_losses, "val": clip_val_losses, "bleu4": clip_bleu}, f)
else:
    clip_train_losses, clip_val_losses, clip_bleu = train_one_model(
        clip_enc, clip_dec, True, "CLIP + Att", CKPT_CLIP
    )
    with open(_curve_path, "w") as f:
        json.dump({"train": clip_train_losses, "val": clip_val_losses, "bleu4": clip_bleu}, f)


In [ ]:
model_curves = {
    "Baseline (no att)": (base_train_losses, base_val_losses, base_bleu, "tomato"),
    "ResNet-50 + Att":   (att_train_losses,  att_val_losses,  att_bleu,  "steelblue"),
    "ViT-B/16 + Att":    (vit_train_losses,  vit_val_losses,  vit_bleu,  "mediumpurple"),
    "CLIP + Att":        (clip_train_losses, clip_val_losses, clip_bleu, "crimson"),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, (tr, val, bl, color) in model_curves.items():
    axes[0].plot(tr,  label=name, color=color)
    axes[1].plot(val, label=name, color=color)
    axes[2].plot(bl,  label=name, color=color)

for ax, title in zip(axes, ["Training Loss", "Validation Loss", "BLEU-4 (val, beam=3)"]):
    ax.set_xlabel("Epoch"); ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
plt.show()

---
## 8. Evaluation — BLEU & METEOR

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score


def evaluate_model(enc, dec, has_attention, label, beam_size=3):
    enc.eval(); dec.eval()
    refs_list, hyps_list = [], []

    with torch.no_grad():
        for img_name in tqdm(test_images, desc=f"Evaluating {label}"):
            try:
                raw = Image.open(os.path.join(IMG_DIR, img_name)).convert("RGB")
            except Exception:
                continue
            tensor = eval_transform(raw).unsqueeze(0).to(device)
            feat   = enc(tensor)
            if has_attention:
                gen, _ = dec.generate_caption(feat, beam_size=beam_size)
            else:
                gen    = dec.generate_caption(feat, beam_size=beam_size)
            hyps_list.append(" ".join(gen))
            refs_list.append([re.sub(r"[^a-z ]", "", c.lower()).strip()
                              for c in image_to_captions[img_name]])

    smooth   = SmoothingFunction().method1
    refs_tok = [[r.split() for r in rl] for rl in refs_list]
    hyps_tok = [h.split() for h in hyps_list]

    bleu1 = corpus_bleu(refs_tok, hyps_tok, weights=(1,0,0,0),         smoothing_function=smooth)
    bleu2 = corpus_bleu(refs_tok, hyps_tok, weights=(.5,.5,0,0),       smoothing_function=smooth)
    bleu3 = corpus_bleu(refs_tok, hyps_tok, weights=(1/3,1/3,1/3,0),   smoothing_function=smooth)
    bleu4 = corpus_bleu(refs_tok, hyps_tok, weights=(.25,.25,.25,.25),  smoothing_function=smooth)
    try:
        meteor = sum(
            meteor_score([r.split() for r in rl], h.split())
            for rl, h in zip(refs_list, hyps_list)
        ) / len(hyps_list)
    except LookupError:
        print("  [METEOR skipped — wordnet not available]")
        meteor = 0.0

    metrics = {"BLEU-1":bleu1,"BLEU-2":bleu2,"BLEU-3":bleu3,"BLEU-4":bleu4,"METEOR":meteor}
    print(f"--- {label} (beam={beam_size}) ---")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    assert bleu1 >= bleu2 >= bleu3 >= bleu4, f"BLEU ordering violated: {metrics}"
    return metrics


base_metrics = evaluate_model(base_enc, base_dec, False, "Baseline")
att_metrics  = evaluate_model(att_enc,  att_dec,  True,  "ResNet-50 + Att")
vit_metrics  = evaluate_model(vit_enc,  vit_dec,  True,  "ViT-B/16 + Att")
clip_metrics = evaluate_model(clip_enc, clip_dec, True,  "CLIP + Att")

---
## 9. Results Table

In [ ]:
paper_att  = {"BLEU-1":0.707,"BLEU-2":0.492,"BLEU-3":0.344,"BLEU-4":0.243,"METEOR":0.239}
paper_base = {"BLEU-1":0.671,"BLEU-2":0.457,"BLEU-3":0.314,"BLEU-4":0.213,"METEOR":0.217}

results = {
    "Paper — SAT (soft att)":   paper_att,
    "Paper — No attention":     paper_base,
    "Ours — Baseline":          base_metrics,
    "Ours — ResNet-50 + Att":   att_metrics,
    "Ours — ViT-B/16 + Att":    vit_metrics,
    "Ours — CLIP + Att":        clip_metrics,
}

df = pd.DataFrame(results).T[["BLEU-1","BLEU-2","BLEU-3","BLEU-4","METEOR"]].round(4)
our_keys = [k for k in results if k.startswith("Ours")]
best_key = df.loc[our_keys]["BLEU-4"].idxmax()
print(f"Best model by BLEU-4: {best_key}  ->  {df.loc[best_key,'BLEU-4']:.4f}\n")
print(df.to_string())
df.to_csv(os.path.join(OUT_DIR, "results.csv"))
print("\nSaved to", os.path.join(OUT_DIR, "results.csv"))

---
## 10. Export Best Model Bundle

Picks the highest BLEU-4 checkpoint and saves a portable `best_model_bundle.pth`
that includes the vocab — ready for the inference / listing-generator notebook.

In [ ]:
import glob

ckpt_files = glob.glob(os.path.join(OUT_DIR, "best_*.pth"))

best_score, best_meta = -float("inf"), None
for ckpt_path in ckpt_files:
    try:
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        score = ck.get("bleu4", -1.0)
        print(f"  {os.path.basename(ckpt_path)}: BLEU-4 = {score:.4f}  ({ck.get('model_name','')})")
        if score > best_score:
            best_score = score
            best_meta  = ck
    except Exception as e:
        print(f"  Could not read {ckpt_path}: {e}")

assert best_meta is not None, "No checkpoints found — run training cells first."
print(f"\nBest: {best_meta['model_name']}  (BLEU-4 = {best_score:.4f})")

bundle = {
    "encoder_state": best_meta["encoder_state"],
    "decoder_state": best_meta["decoder_state"],
    "has_attention": best_meta["has_attention"],
    "model_name":    best_meta["model_name"],
    "bleu4":         best_score,
    "vocab_size":    VOCAB_SIZE,
    "word_to_idx":   word_to_idx,
    "idx_to_word":   {str(k): v for k, v in idx_to_word.items()},
    "trained_on":    ["Flickr8k"],
    "beam_size":     BEAM_SIZE,
    "embed_dim":     256,
    "decoder_dim":   512,
    "encoder_dim":   512,
}

bundle_path = os.path.join(OUT_DIR, "best_model_bundle.pth")
torch.save(bundle, bundle_path)
print(f"Bundle saved -> {bundle_path}  ({os.path.getsize(bundle_path)/1e6:.1f} MB)")

---
## Done

All outputs are in `outputs/`:
- `best_*.pth` — individual checkpoints per model
- `*_curves.json` — training curves data
- `training_curves.png` — visualization
- `results.csv` — final metrics table
- `vocab.json` — vocabulary
- `best_model_bundle.pth` — **portable bundle for the inference notebook**

Use `best_model_bundle.pth` with the apartment listing generator notebook.